# Nome: Raylander Marques Melo
# Matrícula: 586108

In [11]:
!pip install pandas spacy mlxtend


  Using cached matplotlib-3.10.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (11 kB)
  Using cached contourpy-1.3.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (5.5 kB)
  Using cached cycler-0.12.1-py3-none-any.whl.metadata (3.8 kB)
  Using cached kiwisolver-1.4.8-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl.metadata (6.2 kB)
  Using cached pyparsing-3.2.3-py3-none-any.whl.metadata (5.0 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.4/1.4 MB 5.1 MB/s eta 0:00:0046.7 MB/s eta 0:00:01
Using cached matplotlib-3.10.3-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (8.6 MB)
Using cached contourpy-1.3.2-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (323 kB)
Using cached cycler-0.12.1-py3-none-any.whl (8.3 kB)
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 4.9/4.9 MB 19.0 MB/s eta 0:00:0031m110.5 MB/s eta 0:00:01
Using cached kiwisolver-1.4.8-cp312-cp312-manylinux_2_17_x86_64.manylinux2014_x86_64.whl (1.5 MB

In [13]:
import pandas as pd
import spacy
from mlxtend.frequent_patterns import apriori, association_rules, fpgrowth
from collections import defaultdict
import os
import psutil
import time
from collections import Counter
from mlxtend.preprocessing import TransactionEncoder

In [ ]:
def monitor_memory(threshold_mb=10240, check_interval=5):
    proc = psutil.Process(os.getpid())
    while True:
        mem_mb = proc.memory_info().rss / (1024 ** 2)
        print(f"Memória atual: {mem_mb:.2f} MB")
        if mem_mb > threshold_mb:
            print(f"Memória excedeu {threshold_mb} MB. Encerrando o processo...")
            os._exit(1)
        time.sleep(check_interval)

# Execute essa função em uma thread separada
import threading
threading.Thread(target=monitor_memory, daemon=True).start()


Memória atual: 64.12 MB


Memória atual: 439.24 MB
Memória atual: 523.30 MB
Memória atual: 764.73 MB
Memória atual: 1319.61 MB
Memória atual: 1558.89 MB
Memória atual: 7262.84 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 1338.13 MB
Memória atual: 

# Questão 1:
# Utilizando  os  dados  referente  a  postagens  no  WhatsApp,  descubra  regras  que  associem entidades nomeadas presentes nessas mensagens.
# Para cada mensagem identifique uma lista de entidades nomeadas. Utilizando essas listas de entidades nomeadas descubra regras de associação interessantes. 

## a) Ler o dataset fakeTelegram.BR_2022.csv, o qual está disponível no link a seguir: https://drive.google.com/file/d/1c_hLzk85pYw-huHSnFYZM_gn-dUsYRDm/view?usp=drive_link

In [2]:
# Abre do aquivo CSV e printa alguns elementos
df = pd.read_csv("/home/raylander/Desktop/Introdução a Ciência de Dados/fakeTelegram.BR_2022_tratado.csv")
df

,date_message,id_member_anonymous,id_group_anonymous,media,media_type,media_url,has_media,has_media_url,trava_zap,text_content_anonymous,...,id_message,message_type,messenger,media_name,media_md5,caracteres,words,viral,sharings,sentiment
0,2022-10-05 06:25:04,1078cc958f0febe28f4d03207660715f,12283e08a2eb5789201e105b34489ee7,NaN,NaN,NaN,False,False,False,Então é Fato Renato o áudio que eu ouvi no wha...,...,16385,Texto,telegram,NaN,NaN,110,20,0,1.0,0
1,2022-10-05 06:25:08,NaN,12283e08a2eb5789201e105b34489ee7,NaN,NaN,NaN,False,False,False,"Saiu no YouTube do presidente a 8 horas atrás,...",...,16386,Texto,telegram,NaN,NaN,141,23,0,1.0,1
2,2022-10-05 06:26:28,92a2d8fd7144074f659d1d29dc3751da,9f2d7394334eb224c061c9740b5748fc,NaN,NaN,NaN,False,False,False,"É isso, nossa parte já foi quase toda feita. N...",...,16366,Texto,telegram,NaN,NaN,350,59,0,1.0,-1
3,2022-10-05 06:27:28,d60aa38f62b4977426b70944af4aff72,c8f2de56550ed0bf85249608b7ead93d,94dca4cda503100ebfda7ce2bcc060eb.jpg,image/jpg,NaN,True,False,False,GENTE ACHEI ELES EM UMA SEITA MAÇONÁRICA,...,19281,Imagem,telegram,NaN,94dca4cda503100ebfda7ce2bcc060eb,40,7,0,1.0,0
4,2022-10-05 06:27:44,cd6979b0b5265f08468fa1689b6300ce,e56ec342fc599ebb4ed89655eb6f03aa,5ad5c8bbe9da93a37fecf3e5aa5b0637.jpg,image/jpg,NaN,True,False,False,NaN,...,507185,Imagem,telegram,NaN,5ad5c8bbe9da93a37fecf3e5aa5b0637,0,0,1,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
557581,2022-11-11 12:06:15,333e9869f23dbd4682d1be382d9c1e59,e56ec342fc599ebb4ed89655eb6f03aa,25e43b6a58b848c43ad5b5f9e979822a.jpg,url,https://terrabrasilnoticias.com/2022/11/bndes-...,True,True,False,"BNDES tem lucro de R$ 9,6 bilhões no terceiro ...",...,575796,Url,telegram,NaN,25e43b6a58b848c43ad5b5f9e979822a,152,12,1,2.0,1
557582,2022-11-11 12:09:08,NaN,5b10d7739171149be6d9961e3350c071,657949d03e4088f6b332e2686ccd3221.jpg,url,https://youtu.be/8g1Vz9_0xVk,True,True,False,https://youtu.be/8g1Vz9_0xVk,...,1286443,Url,telegram,NaN,657949d03e4088f6b332e2686ccd3221,28,1,1,2.0,0
557583,2022-11-11 12:09:47,NaN,1590a03f43b5ba4b6147a1c5e1dd357b,a21848a61045380a6483866daed0ca0e.jpg,image/jpg,https://t.me/vemprasruas,True,True,False,"Empresários, demitam os petistas primeiro.\n\n...",...,13294,Imagem,telegram,NaN,a21848a61045380a6483866daed0ca0e,68,6,1,4.0,0
557584,2022-11-11 12:09:46,NaN,5b10d7739171149be6d9961e3350c071,a21848a61045380a6483866daed0ca0e.jpg,image/jpg,https://t.me/vemprasruas,True,True,False,"Empresários, demitam os petistas primeiro.\n\n...",...,1286444,Imagem,telegram,NaN,a21848a61045380a6483866daed0ca0e,68,6,1,4.0,0


In [3]:
# Soma a quantidade de elementos com True na coluna 'trava_zap' já existente no dataset
print(df['trava_zap'].sum())

16


In [4]:
# Retira as linhas que possuem trava-zap
df = df[df['trava_zap'] == False]
# Printa o dataset resultante
df

,date_message,id_member_anonymous,id_group_anonymous,media,media_type,media_url,has_media,has_media_url,trava_zap,text_content_anonymous,...,id_message,message_type,messenger,media_name,media_md5,caracteres,words,viral,sharings,sentiment
0,2022-10-05 06:25:04,1078cc958f0febe28f4d03207660715f,12283e08a2eb5789201e105b34489ee7,NaN,NaN,NaN,False,False,False,Então é Fato Renato o áudio que eu ouvi no wha...,...,16385,Texto,telegram,NaN,NaN,110,20,0,1.0,0
1,2022-10-05 06:25:08,NaN,12283e08a2eb5789201e105b34489ee7,NaN,NaN,NaN,False,False,False,"Saiu no YouTube do presidente a 8 horas atrás,...",...,16386,Texto,telegram,NaN,NaN,141,23,0,1.0,1
2,2022-10-05 06:26:28,92a2d8fd7144074f659d1d29dc3751da,9f2d7394334eb224c061c9740b5748fc,NaN,NaN,NaN,False,False,False,"É isso, nossa parte já foi quase toda feita. N...",...,16366,Texto,telegram,NaN,NaN,350,59,0,1.0,-1
3,2022-10-05 06:27:28,d60aa38f62b4977426b70944af4aff72,c8f2de56550ed0bf85249608b7ead93d,94dca4cda503100ebfda7ce2bcc060eb.jpg,image/jpg,NaN,True,False,False,GENTE ACHEI ELES EM UMA SEITA MAÇONÁRICA,...,19281,Imagem,telegram,NaN,94dca4cda503100ebfda7ce2bcc060eb,40,7,0,1.0,0
4,2022-10-05 06:27:44,cd6979b0b5265f08468fa1689b6300ce,e56ec342fc599ebb4ed89655eb6f03aa,5ad5c8bbe9da93a37fecf3e5aa5b0637.jpg,image/jpg,NaN,True,False,False,NaN,...,507185,Imagem,telegram,NaN,5ad5c8bbe9da93a37fecf3e5aa5b0637,0,0,1,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
557581,2022-11-11 12:06:15,333e9869f23dbd4682d1be382d9c1e59,e56ec342fc599ebb4ed89655eb6f03aa,25e43b6a58b848c43ad5b5f9e979822a.jpg,url,https://terrabrasilnoticias.com/2022/11/bndes-...,True,True,False,"BNDES tem lucro de R$ 9,6 bilhões no terceiro ...",...,575796,Url,telegram,NaN,25e43b6a58b848c43ad5b5f9e979822a,152,12,1,2.0,1
557582,2022-11-11 12:09:08,NaN,5b10d7739171149be6d9961e3350c071,657949d03e4088f6b332e2686ccd3221.jpg,url,https://youtu.be/8g1Vz9_0xVk,True,True,False,https://youtu.be/8g1Vz9_0xVk,...,1286443,Url,telegram,NaN,657949d03e4088f6b332e2686ccd3221,28,1,1,2.0,0
557583,2022-11-11 12:09:47,NaN,1590a03f43b5ba4b6147a1c5e1dd357b,a21848a61045380a6483866daed0ca0e.jpg,image/jpg,https://t.me/vemprasruas,True,True,False,"Empresários, demitam os petistas primeiro.\n\n...",...,13294,Imagem,telegram,NaN,a21848a61045380a6483866daed0ca0e,68,6,1,4.0,0
557584,2022-11-11 12:09:46,NaN,5b10d7739171149be6d9961e3350c071,a21848a61045380a6483866daed0ca0e.jpg,image/jpg,https://t.me/vemprasruas,True,True,False,"Empresários, demitam os petistas primeiro.\n\n...",...,1286444,Imagem,telegram,NaN,a21848a61045380a6483866daed0ca0e,68,6,1,4.0,0


In [5]:
# Remover linhas duplicadas com base em todas as colunas
df = df.drop_duplicates()
df

,date_message,id_member_anonymous,id_group_anonymous,media,media_type,media_url,has_media,has_media_url,trava_zap,text_content_anonymous,...,id_message,message_type,messenger,media_name,media_md5,caracteres,words,viral,sharings,sentiment
0,2022-10-05 06:25:04,1078cc958f0febe28f4d03207660715f,12283e08a2eb5789201e105b34489ee7,NaN,NaN,NaN,False,False,False,Então é Fato Renato o áudio que eu ouvi no wha...,...,16385,Texto,telegram,NaN,NaN,110,20,0,1.0,0
1,2022-10-05 06:25:08,NaN,12283e08a2eb5789201e105b34489ee7,NaN,NaN,NaN,False,False,False,"Saiu no YouTube do presidente a 8 horas atrás,...",...,16386,Texto,telegram,NaN,NaN,141,23,0,1.0,1
2,2022-10-05 06:26:28,92a2d8fd7144074f659d1d29dc3751da,9f2d7394334eb224c061c9740b5748fc,NaN,NaN,NaN,False,False,False,"É isso, nossa parte já foi quase toda feita. N...",...,16366,Texto,telegram,NaN,NaN,350,59,0,1.0,-1
3,2022-10-05 06:27:28,d60aa38f62b4977426b70944af4aff72,c8f2de56550ed0bf85249608b7ead93d,94dca4cda503100ebfda7ce2bcc060eb.jpg,image/jpg,NaN,True,False,False,GENTE ACHEI ELES EM UMA SEITA MAÇONÁRICA,...,19281,Imagem,telegram,NaN,94dca4cda503100ebfda7ce2bcc060eb,40,7,0,1.0,0
4,2022-10-05 06:27:44,cd6979b0b5265f08468fa1689b6300ce,e56ec342fc599ebb4ed89655eb6f03aa,5ad5c8bbe9da93a37fecf3e5aa5b0637.jpg,image/jpg,NaN,True,False,False,NaN,...,507185,Imagem,telegram,NaN,5ad5c8bbe9da93a37fecf3e5aa5b0637,0,0,1,NaN,0
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
557581,2022-11-11 12:06:15,333e9869f23dbd4682d1be382d9c1e59,e56ec342fc599ebb4ed89655eb6f03aa,25e43b6a58b848c43ad5b5f9e979822a.jpg,url,https://terrabrasilnoticias.com/2022/11/bndes-...,True,True,False,"BNDES tem lucro de R$ 9,6 bilhões no terceiro ...",...,575796,Url,telegram,NaN,25e43b6a58b848c43ad5b5f9e979822a,152,12,1,2.0,1
557582,2022-11-11 12:09:08,NaN,5b10d7739171149be6d9961e3350c071,657949d03e4088f6b332e2686ccd3221.jpg,url,https://youtu.be/8g1Vz9_0xVk,True,True,False,https://youtu.be/8g1Vz9_0xVk,...,1286443,Url,telegram,NaN,657949d03e4088f6b332e2686ccd3221,28,1,1,2.0,0
557583,2022-11-11 12:09:47,NaN,1590a03f43b5ba4b6147a1c5e1dd357b,a21848a61045380a6483866daed0ca0e.jpg,image/jpg,https://t.me/vemprasruas,True,True,False,"Empresários, demitam os petistas primeiro.\n\n...",...,13294,Imagem,telegram,NaN,a21848a61045380a6483866daed0ca0e,68,6,1,4.0,0
557584,2022-11-11 12:09:46,NaN,5b10d7739171149be6d9961e3350c071,a21848a61045380a6483866daed0ca0e.jpg,image/jpg,https://t.me/vemprasruas,True,True,False,"Empresários, demitam os petistas primeiro.\n\n...",...,1286444,Imagem,telegram,NaN,a21848a61045380a6483866daed0ca0e,68,6,1,4.0,0


In [4]:
mensagens = df['text_content_anonymous'].astype(str).tolist()

# Carregando modelo de português
nlp = spacy.load("pt_core_news_lg")  # ou pt_core_news_sm se for mais leve

def extrair_entidades(mensagem):
    doc = nlp(mensagem)
    return [ent.text for ent in doc.ents]

# Lista de listas com entidades por mensagem
entidades_por_mensagem = [extrair_entidades(msg) for msg in mensagens]


NameError: name 'df' is not defined

In [12]:
df_entidades = pd.DataFrame({
    "mensagem": mensagens,
    "entidades": [", ".join(ent) for ent in entidades_por_mensagem]
})

df_entidades.to_csv("entidades_por_mensagem.csv", index=False)

In [3]:
# Lendo o arquivo CSV salvo
df_entidades = pd.read_csv("entidades_por_mensagem.csv")

# Tratar NaNs: substitui por string vazia antes do split
df_entidades["entidades"] = df_entidades["entidades"].fillna("").apply(lambda x: x.split(", ") if x else [])

# Acessar como lista de listas, se necessário
entidades_por_mensagem = df_entidades["entidades"].tolist()


In [ ]:
# Contar todas as entidades
contador = Counter(ent for msg in entidades_por_mensagem for ent in msg)

# Manter apenas as entidades que aparecem em pelo menos 5 mensagens
entidades_validas = {ent for ent, count in contador.items() if count >= 5}

# Filtrar
entidades_filtradas = [
    [ent for ent in msg if ent in entidades_validas]
    for msg in entidades_por_mensagem
]

# Remover mensagens vazias (sem entidades após filtragem)
entidades_filtradas = [msg for msg in entidades_filtradas if msg]


In [ ]:
te = TransactionEncoder()
te_ary = te.fit(entidades_filtradas).transform(entidades_filtradas)
df_transacoes = pd.DataFrame(te_ary, columns=te.columns_)


In [ ]:
# Apriori
frequent_apriori = apriori(df_transacoes, min_support=0.05, use_colnames=True)
regras_apriori = association_rules(frequent_apriori, metric="confidence", min_threshold=0.5)

In [ ]:
# Feito com entidades que aparecem mais de 5 vezes
print(f"Apriori: {len(regras_apriori)} regras")
print(regras_apriori)

Apriori: 12 regras
                                          antecedents  \
0                                               (TSE)   
1              (This community was blocked in Brazil)   
2                                               (TSE)   
3                   (of the Superior Electoral Court)   
4                   (of the Superior Electoral Court)   
5              (This community was blocked in Brazil)   
6              (TSE, of the Superior Electoral Court)   
7         (TSE, This community was blocked in Brazil)   
8   (of the Superior Electoral Court, This communi...   
9                                               (TSE)   
10                  (of the Superior Electoral Court)   
11             (This community was blocked in Brazil)   

                                          consequents  antecedent support  \
0              (This community was blocked in Brazil)            0.086288   
1                                               (TSE)            0.060816   
2       

In [9]:
# FPGrowth
frequent_fpgrowth = fpgrowth(df_transacoes, min_support=0.05, use_colnames=True)
regras_fpgrowth = association_rules(frequent_fpgrowth, metric="confidence", min_threshold=0.5)


In [10]:
print(f"FP-Growth: {len(regras_fpgrowth)} regras")
print(regras_fpgrowth)

FP-Growth: 13 regras
                                          antecedents  \
0                                           (Welcome)   
1                                               (TSE)   
2                   (of the Superior Electoral Court)   
3              (This community was blocked in Brazil)   
4                   (of the Superior Electoral Court)   
5              (This community was blocked in Brazil)   
6                                               (TSE)   
7         (This community was blocked in Brazil, TSE)   
8   (This community was blocked in Brazil, of the ...   
9              (TSE, of the Superior Electoral Court)   
10             (This community was blocked in Brazil)   
11                                              (TSE)   
12                  (of the Superior Electoral Court)   

                                          consequents  antecedent support  \
0                                              (USER)            0.060908   
1                   (of th

In [ ]:
# ECLAT
def eclat(transactions, min_support=0.03):
    total = len(transactions)
    min_count = int(total * min_support)
    
    itemsets = defaultdict(set)

    for tid, transaction in enumerate(transactions):
        for item in set(transaction):
            itemsets[frozenset([item])].add(tid)
    
    def recursive_eclat(prefix, items):
        results = []
        for i in range(len(items)):
            item_i = items[i]
            tids_i = itemsets[item_i]
            new_itemset = prefix.union(item_i)
            support = len(tids_i)
            if support >= min_count:
                results.append((set(new_itemset), support))
                suffix = items[i+1:]
                new_items = []
                for item_j in suffix:
                    tids_j = itemsets[item_j]
                    tids_intersection = tids_i & tids_j
                    if len(tids_intersection) >= min_count:
                        itemsets[item_i.union(item_j)] = tids_intersection
                        new_items.append(item_i.union(item_j))
                results.extend(recursive_eclat(new_itemset, new_items))
        return results
    
    return recursive_eclat(set(), list(itemsets.keys()))

# Aplicando ECLAT
eclat_result = eclat(entidades_por_mensagem, min_support=0.03)

# Convertendo para DataFrame
eclat_df = pd.DataFrame(eclat_result, columns=["itemset", "support"])
eclat_df["support"] = eclat_df["support"] / len(entidades_por_mensagem)


In [ ]:
# Feito com entidades que aparecem mais de 5 vezes
print(f"ECLAT: {len(eclat_df)} itemsets frequentes")
print(eclat_df)

ECLAT: 10 itemsets frequentes
                                             itemset   support
0                                        {Bolsonaro}  0.050311
1                                           {Brasil}  0.040334
2                                             {USER}  0.091104
3                                              {TSE}  0.044333
4        {TSE, This community was blocked in Brazil}  0.031246
5  {TSE, This community was blocked in Brazil, of...  0.031246
6             {TSE, of the Superior Electoral Court}  0.031246
7             {This community was blocked in Brazil}  0.031246
8  {This community was blocked in Brazil, of the ...  0.031246
9                  {of the Superior Electoral Court}  0.031246
